# Notebook 3: Digital twins for CO2-aware datacenter operation

Welcome to notebook 3. So far, we investigated the effects of digital-twin-aided autoscaling on ICT infrastructure (notebook 1), and how OpenDT keeps predictions accurate with on-the-fly calibration (notebook 2). In this notebook, we move from performance to sustainability, specifically to energy and CO2 emissions.

Datacenters draw their electricity from the grid of the country they operate in and the same energy can lead to very different CO2 emissions, depending on how the grid produces it. For example, energy produced with coal and gas (also called "brown sources") has a higher CO2 footprint than energy produced with wind or solar power ("green sources"). Digital twins can act as advisors-on-the-side for where to run a workload, CO2-awarely.

OpenDT predicts the CO2 emissions of the same workload, on identical hosts, in different locations, while a human-in-the-loop oversees at least the major decisions. One such major decision can be whether to migrate a workload from one country where the datacenter operates to another such country. In this notebook, focused on CO2-aware workload migration, you will learn:

1. What carbon intensity is, and how it differs across countries and over time;
2. How location influences CO2 emissions, for the same workload and the same hosts;
3. How much CO2 a workload saves by following the greenest grid, hour by hour;
4. Where the trade-offs are: what the human-in-the-loop weighs before migrating a workload.

By the end of the notebook, you will be able to quantify the CO2 emissions of a workload in different locations, and reason, with numbers, about the trade-off between CO2 emissions and performance when choosing where to run a workload.


_For the best learning experience, we strongly recommend starting with notebook 1._

> **Scenario:** you are managing the ICT infrastructure of a company which operates datacenters in four European countries: (1) the Netherlands (NL), (2) Germany (DE), (3) Romania (RO), and (4) Poland (PL), each with hosts identical to those of SURF. The company runs the workload in NL; with OpenDT as an advisor-on-the-side, you decide whether, and when, to migrate it to another of the company's datacenters.

> Run the cells in order (Shift+Enter). For each question, click your selected option to reveal the answer.

In [ ]:
import pandas as pd
from resources import *

check_setup()

# 1. What is carbon intensity, and how does it differ across countries?

**Carbon intensity** = the CO2 emitted per unit of electricity, in CO2 equivalents [gCO2/kWh]; it depends on how the grid produces its electricity (e.g., coal, gas, wind, solar), and changes from hour to hour.

**CO2 emissions** = the energy a datacenter consumes, multiplied by the carbon intensity of its grid at the time of consumption:

$\text{CO}_2\,[\text{g}] = \sum_{t} E_t \times CI_t$, where
- $E_t$ is the energy consumed in each sample [kWh],
- $CI_t$ is the carbon intensity of the grid at that time [gCO2/kWh].

**Workload migration** = moving a workload (i.e., its jobs and their data) from one datacenter to another (e.g., in a country with a greener grid, i.e., with a lower carbon intensity).

In this notebook, we compare four grids from Europe, each with distinct electricity sources: the Netherlands (NL, where SURF operates), Germany (DE), Romania (RO), and Poland (PL). We match the hourly carbon intensity of the grids with the SURF week (6-13 October 2022), from Electricity Maps ([ODbL](../workload/SURF/carbon_zones/LICENSE.md)).

#### The carbon intensity of a grid follows how the country produces its electricity; in 2022, the four countries produce this energy from very different sources ([IEA](https://www.iea.org/data-and-statistics/data-tools/energy-statistics-data-browser))

In [ ]:
plot_electricity_mix();  # share of each source in the electricity generation of 2022 (IEA)

In [ ]:
WEEK = slice(TRACE_START, TRACE_END - pd.Timedelta("1h"))  # the 168 hours of the SURF week
intensity = pd.DataFrame({zone: carbon_intensity(zone)[WEEK] for zone in CARBON_ZONES})  # [gCO2/kWh]

ax = intensity.plot()
ax.set(xlabel="Time [UTC]", ylabel="Carbon Intensity [gCO2/kWh]", ylim=(0, None))
ax.yaxis.set_major_formatter("{x:,.0f}")
legend_above(ax);

In [ ]:
pd.DataFrame([intensity.mean()], index=["Average [gCO2/kWh]"]).style.format(precision=0)

In [ ]:
ask_question("3.1")

# 2. How much CO2 does the same workload emit, in each country?

As a what-if analysis, OpenDT simulates the full SURF week once per country, with the same workload and the same 277 hosts.

In [ ]:
ask_question("3.2")

In [ ]:
runs = simulate_many("carbon", CARBON_ZONES, until=TRACE_END)  # one simulation of the full week per country

table = pd.DataFrame([run.summary() for run in runs.values()], index=list(runs))
rows = {"energy_kwh": "Energy [kWh]", "co2_kg": "CO2 Emissions [kg]"}
table[list(rows)].T.rename(index=rows).style.format(precision=0, thousands=",")

In [ ]:
ask_question("3.3")

# 3. Does the greenest grid stay the greenest, throughout the day?

We now look at the carbon intensity window by window, on Monday 10 October, from 06:00 to 19:00 UTC, each window of 1 hour; the greenest grid of each window is in bold.

In [ ]:
hours = slice("2022-10-10 06:00", "2022-10-10 18:00")  # 13 windows, 06:00 to 19:00 UTC on 10 October
monday = intensity.loc[hours].copy()
monday.index = [f"{h:%H:%M}-{h + pd.Timedelta('1h'):%H:%M}" for h in monday.index]
monday.style.format(precision=0).highlight_min(axis=1, props="font-weight: bold")

In [ ]:
ask_question("3.4")


A digital twin could advise migrating the workload, every hour, to the country with the greenest grid, since the company has a datacenter in each country; we replay this advice over the whole week, from the simulations above. However, we do not take into account the migration costs (e.g., transfer, boot-up, reserving), which would also influence the total CO2 emissions.

In [ ]:
emissions = pd.DataFrame({zone: run.power["carbon_emission"] for zone, run in runs.items()})  # [g]
co2_per_hour = emissions[emissions.index < TRACE_END].resample("1h").sum() / 1000  # [kg], the 168 hours
greenest = co2_per_hour.idxmin(axis=1)  # the country with the greenest grid in each hour
migrations = (greenest != greenest.shift())[1:].sum()  # changes of country from one hour to the next

print(f"NL greener than RO: {(intensity['NL'] < intensity['RO']).sum()} out of {len(intensity)} hours")
print(f"Deploying only in NL:  {co2_per_hour['NL'].sum():,.0f} kg CO2")
print(f"Deploying only in RO:  {co2_per_hour['RO'].sum():,.0f} kg CO2")
greenest_kg = co2_per_hour.min(axis=1).sum()
print(f"Deploying in greenest: {greenest_kg:,.0f} kg CO2, with {migrations} migrations, and evaluated every hour")

In [ ]:
ask_question("3.5")

# 4. Where are the trade-offs of CO2-aware workload migration?

OpenDT operates autonomously, yet a human can (optionally) intervene and dictate major decisions; migrating a workload to another country is one such major decision.

In [ ]:
ask_question("3.6")

#### 4.1 Trade-off between CO2 and performance, in the number of hosts
In notebook 1, we downscaled SURF to fewer hosts; here, we combine downscaling with a migration to RO, the grid with the lowest carbon intensity of the four, on average.

In [ ]:
options = {"NL, 277 hosts": runs["NL"],
           "NL, 200 hosts": simulate(until=TRACE_END, hosts=200, carbon="NL"),
           "RO, 200 hosts": simulate(until=TRACE_END, hosts=200, carbon="RO"),
           "RO, 138 hosts": simulate(until=TRACE_END, hosts=138, carbon="RO")}
compare = pd.DataFrame([run.summary() for run in options.values()], index=list(options))
rows = {"co2_kg": "CO2 Emissions [kg]", "jobs_waited": "Jobs that Waited", "mean_wait_min": "Mean Wait [min]"}
compare[list(rows)].T.rename(index=rows).style.format(precision=0, thousands=",")

In [ ]:
ask_question("3.7")

#### 4.2 One more trade-off (your turn)
Can you identify another trade-off introduced by CO2-aware workload migration?

<details><summary><b>Hint</b></summary>What does it take to move SURF's jobs and their data to a datacenter in another country, and who notices?
<details><summary><b>Answer</b></summary>Before migrating, the human-in-the-loop considers, e.g., (1) the energy and time to move data and jobs over the network, (2) the latency for the users, (3) the rules on where data may be stored, (4) the free capacity and the cost of the remote datacenter, and (5) the CO2 emitted to build new hardware.</details>
</details>

# 5. What if the company would also operate in Sweden, the greenest grid in Europe, in Oct 2022?

Sweden (SE) has the greenest grid in Europe: 25 gCO2/kWh on average in 2022 (Electricity Maps), as it produced 40% of its electricity from hydropower, 30% from nuclear power, and 19% from wind (IEA). What if the company also operated a datacenter in SE, with the same hosts?

In [ ]:
ask_question("3.8")

In [ ]:
sweden = simulate(until=TRACE_END, carbon="SE")  # the full week, on the greenest grid in Europe
co2_kg = {**table["co2_kg"], "Greenest grid every hour": greenest_kg, "SE": sweden.summary()["co2_kg"]}
pd.DataFrame([co2_kg], index=["CO2 Emissions [kg]"]).style.format(precision=0, thousands=",")